In [1]:
from pathlib import Path
from google.colab import drive
import sys
import importlib

drive.mount("/content/drive")

# Clone the repository to this location, or change REPO_ROOT to your checkout.
REPO_ROOT = Path("/content/drive/MyDrive/jev_project/jev-like-decision-tuning")
assert (REPO_ROOT / "pyproject.toml").is_file(), "Set REPO_ROOT to the repository checkout"
%pip install -q -e "$REPO_ROOT"

source_root = REPO_ROOT / "src"

for filename in ("__init__.py", "data.py", "prompts.py"):
    source_file = source_root / "jev_decision_tuning" / filename
    assert source_file.is_file(), f"Missing source file: {source_file}"

if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

importlib.invalidate_caches()
from jev_decision_tuning.data import (
    load_sentiment_dataset,
    sample_examples,
    sample_balanced,
    format_training_dataset,
    tokenize_dataset,
)
from jev_decision_tuning.prompts import (
    INSTRUCTION,
    LABEL_NAMES,
    LETTERS,
    build_prompt,
    format_example,
    render_chat_prompt,
)

label_names = list(LABEL_NAMES)
letters = list(LETTERS)

Mounted at /content/drive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for jev-like-decision-tuning (pyproject.toml) ... done


In [2]:
ds = load_sentiment_dataset(
    "/content/drive/MyDrive/jev_project/data/tweet_eval_sentiment"
)

## Format a sentiment example

In [3]:
instance = ds["train"][0]
result = format_example(
    instance,
    compact_options=True,
    include_reply_instruction=False,
)
print(result["prompt"])
print(result["target_letter"])

Classify the sentiment of the following text.

Text: "QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"

A:negative
B:neutral
C:positive

Answer:
C


In [4]:
%pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.3 MB/s eta 0:00:00


## Load the model

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
)

model.eval()

print("Model device:", model.device)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model device: cuda:0


## Evaluate the base model on 100 validation examples

In [6]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# 1. Fix a 100-example subset for before/after comparisons.
samples = sample_examples(ds["validation"], count=100, seed=42)

labels = list(LABEL_NAMES)

encoded = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in letters
]
assert all(len(ids) == 1 for ids in encoded)
option_ids = [ids[0] for ids in encoded]


# 2. Score one example with the original compact option spacing.
def predict_one(example):
    prompt = build_prompt(example["text"], compact_options=True)
    text = render_chat_prompt(tokenizer, prompt)

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model(**inputs)
        scores = outputs.logits[0, -1, option_ids].float()
        probs = torch.softmax(scores, dim=-1).cpu().tolist()

    return probs


# 3. Run predictions and retain the results.
model.eval()
records = []

for example in tqdm(samples):
    probs = predict_one(example)
    prediction = max(range(3), key=lambda i: probs[i])

    records.append({
        "text": example["text"],
        "true_label": example["label"],
        "pred_label": prediction,
        "p_negative": probs[0],
        "p_neutral": probs[1],
        "p_positive": probs[2],
    })

results = pd.DataFrame(records)

# 4. Compute metrics.
y_true = results["true_label"]
y_pred = results["pred_label"]

print(f"\nAccuracy:{accuracy_score(y_true, y_pred):.1%}")

print("\nPer-class performance:")
print(classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    target_names=labels,
    digits=3,
    zero_division=0,
))

print("Confusion matrix (rows=true, columns=predicted):")
display(pd.DataFrame(
    confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels],
))

  0%|          | 0/100 [00:00<?, ?it/s]


Accuracy:47.0%

Per-class performance:
              precision    recall  f1-score   support

    negative      0.222     1.000     0.364        10
     neutral      0.800     0.087     0.157        46
    positive      0.660     0.750     0.702        44

    accuracy                          0.470       100
   macro avg      0.561     0.612     0.408       100
weighted avg      0.681     0.470     0.417       100

Confusion matrix (rows=true, columns=predicted):


,pred_negative,pred_neutral,pred_positive
true_negative,10,0,0
true_neutral,25,4,17
true_positive,10,1,33


## Fine-tune the model with LoRA

In [7]:
from collections import Counter

# Sample 3,000 examples per class: 9,000 training examples in total.
train_small = sample_balanced(ds["train"], per_class=3000, seed=42)

print("Label order:", label_names)
print("Training examples:", len(train_small))
counts = Counter(train_small["label"])
for label_id, label_name in enumerate(label_names):
    print(f"{label_name}: {counts[label_id]}")
print("First example:", train_small[0])

Filter:   0%|          | 0/45615 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45615 [00:00<?, ? examples/s]

Filter:   0%|          | 0/45615 [00:00<?, ? examples/s]

Label order: ['negative', 'neutral', 'positive']
Training examples: 9000
negative: 3000
neutral: 3000
positive: 3000
First example: {'text': '"Today, while in my American history class I was reading on Islam in Africa. I\'ll probably fail my test on the Industrial Revolution on Mon."', 'label': 0}


In [8]:
# Option shuffling and target remapping use a fresh local RNG on each call.
train_formatted = format_training_dataset(train_small, seed=42)

for i in range(3):
    example = train_formatted[i]
    print(example["prompt"])
    print("Correct option:", example["target_letter"])
    print("Original class:", label_names[example["label"]])
    print()

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Classify the sentiment of the following text.

Text: "Today, while in my American history class I was reading on Islam in Africa. I'll probably fail my test on the Industrial Revolution on Mon."

A: neutral
B: negative
C: positive

Answer:
Reply with only A, B, or C.
Correct option: B
Original class: negative

Classify the sentiment of the following text.

Text: @user Can't say I was overly comfy with it in the mall first Fri afternoon in Eid. So many men. No women. Tried to shrink bust inwards.

A: positive
B: neutral
C: negative

Answer:
Reply with only A, B, or C.
Correct option: B
Original class: neutral

Classify the sentiment of the following text.

Text: My family sat there an talked about how gay rights is bad .-. Guess ima bring a girl home

A: neutral
B: positive
C: negative

Answer:
Reply with only A, B, or C.
Correct option: C
Original class: negative



In [9]:
train_tokenized = tokenize_dataset(train_formatted, tokenizer)

lengths = [len(ids) for ids in train_tokenized["input_ids"]]
print("Examples:", len(train_tokenized))
print("Maximum token count:", max(lengths))
print("Mean token count:", round(sum(lengths) / len(lengths), 1))
print("First decoded input:")
print(tokenizer.decode(train_tokenized[0]["input_ids"]))
print("Correct option:", letters[train_tokenized[0]["labels"]])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Examples: 9000
Maximum token count: 132
Mean token count: 75.7
First decoded input:
<|im_start|>user
Classify the sentiment of the following text.

Text: "Today, while in my American history class I was reading on Islam in Africa. I'll probably fail my test on the Industrial Revolution on Mon."

A: neutral
B: negative
C: positive

Answer:
Reply with only A, B, or C.<|im_end|>
<|im_start|>assistant
<think>

</think>


Correct option: B


In [10]:
%pip install -q -U peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 21.7 MB/s eta 0:00:00


In [11]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [12]:
from peft import LoraConfig, get_peft_model
from transformers import set_seed

set_seed(42)

target_modules = [
    # Attention modules
    "q_proj", "k_proj", "v_proj", "o_proj",

    # MLP modules
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    target_modules=target_modules,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

model.print_trainable_parameters()

trainable params: 8,716,288 || all params: 1,729,291,264 || trainable%: 0.5040


In [13]:
import torch
from torch.utils.data import DataLoader

# Right-pad inputs of different lengths.
tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def collate_batch(examples):
    # Keep the target answer positions separate.
    labels = torch.tensor(
        [example["labels"] for example in examples],
        dtype=torch.long,
    )

    # Collect model inputs.
    features = [
        {
            "input_ids": example["input_ids"],
            "attention_mask": example["attention_mask"],
        }
        for example in examples
    ]

    # Pad inputs to the same length within this batch.
    batch = tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt",
    )

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_tokenized,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(42),
)

batch = next(iter(train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Target option indices:", batch["labels"])
print("Actual input lengths:", batch["attention_mask"].sum(dim=1))

Input shape: torch.Size([4, 87])
Target option indices: tensor([0, 2, 1, 0])
Actual input lengths: tensor([84, 66, 82, 87])


In [14]:
print("Training examples:", len(train_tokenized))
print("Batch size:", train_loader.batch_size)
print("Total batches:", len(train_loader))
model.print_trainable_parameters()

Training examples: 9000
Batch size: 4
Total batches: 2250
trainable params: 8,716,288 || all params: 1,729,291,264 || trainable%: 0.5040


In [15]:
import json
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import set_seed

set_seed(42)
device = model.device

# Use a separate output directory for each run.
save_dir = (
    Path("/content/drive/MyDrive/jev_project")
    / f"lora_attention_mlp_{time.strftime('%Y%m%d_%H%M%S')}"
)

assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"
save_dir.mkdir(parents=True, exist_ok=False)

# Train only LoRA parameters and keep trainable parameters in float32.
trainable_params = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        assert "lora_" in name, f"Non-LoRA parameter is trainable:{name}"
        parameter.data = parameter.data.float()
        trainable_params.append(parameter)

# Token IDs for A, B, and C
encoded_options = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in ["A", "B", "C"]
]
assert all(len(ids) == 1 for ids in encoded_options)

option_ids = torch.tensor(
    [ids[0] for ids in encoded_options],
    device=device,
)

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=5e-5,
    weight_decay=0.01,
)
scaler = torch.amp.GradScaler("cuda", init_scale=1024)

# Reset shuffling so inspecting a batch does not change training order.
train_loader.generator.manual_seed(42)
assert tokenizer.padding_side == "right"

model.config.use_cache = False
model.train()

history = []
window_losses = []
skipped_updates = 0

torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
start = time.perf_counter()

progress = tqdm(train_loader, desc="Training epoch 1/1")

for step, batch in enumerate(progress, start=1):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    targets = batch["labels"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        # Select the last non-padding token for each input.
        last_positions = attention_mask.sum(dim=1) - 1
        row_ids = torch.arange(input_ids.shape[0], device=device)

        scores = outputs.logits[row_ids, last_positions]
        option_scores = scores[:, option_ids].float()

        # Targets are A/B/C positions; compute cross-entropy over the three options.
        loss = F.cross_entropy(option_scores, targets)

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at batch {step}; stopping training")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()

    skipped = scaler.get_scale() < old_scale
    skipped_updates += int(skipped)

    loss_value = loss.item()
    history.append({
        "step": step,
        "loss": loss_value,
        "update_skipped": skipped,
    })
    window_losses.append(loss_value)

    # Report the mean loss over each 50-batch window.
    if step % 50 == 0:
        mean_loss = sum(window_losses) / len(window_losses)
        tqdm.write(f"Batch {step}/{len(train_loader)} | Mean loss over last 50 batches:{mean_loss:.4f}")
        window_losses.clear()

    del outputs, scores, option_scores, loss

optimizer.zero_grad(set_to_none=True)
model.eval()

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

# Save the trained LoRA adapter and tokenizer.
model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

# Save the experiment configuration and loss history.
record = {
    "base_model": "Qwen/Qwen3-1.7B",
    "seed": 42,
    "samples": len(train_tokenized),
    "batch_size": train_loader.batch_size,
    "epochs": 1,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "loss": "cross_entropy_over_A_B_C",
    "skipped_updates": skipped_updates,
    "elapsed_seconds": elapsed,
    "history": history,
}

with open(save_dir / "training_log.json", "w", encoding="utf-8") as f:
    json.dump(record, f, ensure_ascii=False, indent=2)

print(f"\nTraining completed in {elapsed / 60:.1f} minutes")
print(f"Skipped updates:{skipped_updates}")
print(f"Peak GPU memory:{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
print("Saved to:", save_dir)

Training epoch 1/1:   0%|          | 0/2250 [00:00<?, ?it/s]

Batch 50/2250 | Mean loss over last 50 batches:1.8094
Batch 100/2250 | Mean loss over last 50 batches:0.9418
Batch 150/2250 | Mean loss over last 50 batches:0.8196
Batch 200/2250 | Mean loss over last 50 batches:0.7899
Batch 250/2250 | Mean loss over last 50 batches:0.9174
Batch 300/2250 | Mean loss over last 50 batches:0.7603
Batch 350/2250 | Mean loss over last 50 batches:0.7390
Batch 400/2250 | Mean loss over last 50 batches:0.7079
Batch 450/2250 | Mean loss over last 50 batches:0.7964
Batch 500/2250 | Mean loss over last 50 batches:0.6549
Batch 550/2250 | Mean loss over last 50 batches:0.7560
Batch 600/2250 | Mean loss over last 50 batches:0.6889
Batch 650/2250 | Mean loss over last 50 batches:0.7250
Batch 700/2250 | Mean loss over last 50 batches:0.7312
Batch 750/2250 | Mean loss over last 50 batches:0.7817
Batch 800/2250 | Mean loss over last 50 batches:0.7380
Batch 850/2250 | Mean loss over last 50 batches:0.6967
Batch 900/2250 | Mean loss over last 50 batches:0.5507
Batch 950/2

In [16]:
%pip install -q typesafe-sdk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.7 MB/s eta 0:00:00


In [17]:
import json
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from google.colab import userdata
from typesafe_sdk import Choice, TypeSafeClient
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
)

instruction = INSTRUCTION

save_dir = Path("/content/drive/MyDrive/jev_project")/ f"lora_attention_mlp_{time.strftime('%Y%m%d_%H%M%S')}"
jev_dir = Path("/content/drive/MyDrive/jev_project/jev_1.13.0_validation")
assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"
save_dir.mkdir(parents=True, exist_ok=True)
jev_dir.mkdir(parents=True, exist_ok=True)

encoded = [
    tokenizer.encode(letter, add_special_tokens=False)
    for letter in letters
]
assert all(len(ids) == 1 for ids in encoded)
option_ids = [ids[0] for ids in encoded]

In [18]:
def evaluate_validation(
    description,
    backend="qwen",
    dataset=None,
    checkpoint_name="predictions",
):
    """
    backend="qwen": use the current local model.
    backend="jev": call the official Jev API.
    """
    assert backend in ["qwen", "jev"]

    validation = ds["validation"] if dataset is None else dataset
    assert validation.features["label"].names == label_names
    records = {}

    checkpoint = jev_dir / f"{checkpoint_name}.jsonl"

    # Resume Jev evaluation from saved predictions.
    if backend == "jev" and checkpoint.exists():
        with checkpoint.open(encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                record = json.loads(line)
                row_id = record["row_id"]

                assert 0 <= row_id < len(validation)
                assert record["text"] == validation[row_id]["text"]
                assert record["true_label"] == validation[row_id]["label"]
                assert record["model"] == "jev-1.13.0"

                records[row_id] = record

    def save_record(row_id, example, probs, **extra):
        record = {
            "row_id": row_id,
            "text": example["text"],
            "true_label": example["label"],
            "pred_label": max(range(3), key=lambda i: probs[i]),
            "p_negative": probs[0],
            "p_neutral": probs[1],
            "p_positive": probs[2],
            **extra,
        }
        records[row_id] = record
        return record

    if backend == "qwen":
        model.eval()

        for row_id, example in enumerate(
            tqdm(validation, desc=description)
        ):
            prompt = build_prompt(example["text"])
            text = render_chat_prompt(tokenizer, prompt)

            inputs = tokenizer(
                text, return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model(**inputs, use_cache=False)
                scores = outputs.logits[0, -1, option_ids].float()
                probs = torch.softmax(scores, dim=-1).cpu().tolist()

            save_record(row_id, example, probs)
            del outputs

    else:
        with TypeSafeClient(api_key=userdata.get("jev")) as client:
            with checkpoint.open("a", encoding="utf-8") as f:
                with tqdm(
                    total=len(validation),
                    initial=len(records),
                    desc=description,
                ) as progress:

                    for row_id, example in enumerate(validation):
                        if row_id in records:
                            continue

                        response = client.system_one(
                            model="jev-1.13.0",
                            state=example["text"],
                            questions={
                                "sentiment": Choice(
                                    instructions=instruction,
                                    criteria=dict(zip(LETTERS, LABEL_NAMES)),
                                ),
                            },
                        )

                        assert response.model == "jev-1.13.0"
                        answer = response.choices["sentiment"]

                        probs = [
                            float(answer.probabilities[letter])
                            for letter in letters
                        ]

                        record = save_record(
                            row_id,
                            example,
                            probs,
                            api_choice=answer.choice,
                            model=response.model,
                            input_tokens=response.usage.input_tokens,
                            output_tokens=response.usage.output_tokens,
                        )

                        f.write(
                            json.dumps(record, ensure_ascii=False) + "\n"
                        )
                        f.flush()
                        progress.update(1)

    assert set(records) == set(range(len(validation)))

    return pd.DataFrame(
        [records[i] for i in range(len(validation))]
    )

In [21]:
# First, disable LoRA and evaluate the base model.
with model.disable_adapter():
    results_base = evaluate_validation("Base model", backend="qwen")

results_base.to_csv(
    save_dir / "validation_full_base.csv", index=False
)

# Second, restore LoRA and evaluate the fine-tuned model.
results_lora = evaluate_validation("LoRA model", backend="qwen")

results_lora.to_csv(
    save_dir / "validation_full_lora.csv", index=False
)

# Third, evaluate the official Jev model.
results_jev = evaluate_validation("Jev model", backend="jev")

results_jev.to_csv(
    jev_dir / "validation_full_jev.csv", index=False
)

Base model:   0%|          | 0/2000 [00:00<?, ?it/s]

LoRA model:   0%|          | 0/2000 [00:00<?, ?it/s]

Jev model:   0%|          | 0/2000 [00:00<?, ?it/s]

In [22]:
summary = []

for name, frame in [
    ("Base model", results_base),
    ("LoRA model", results_lora),
    ("Jev 1.13.0", results_jev),
]:
    assert frame["row_id"].tolist() == results_base["row_id"].tolist()
    assert frame["true_label"].tolist() == results_base["true_label"].tolist()

    y_true = frame["true_label"]
    y_pred = frame["pred_label"]

    recalls = recall_score(
        y_true, y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    summary.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro-F1": f1_score(
            y_true, y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "negative_recall": recalls[0],
        "neutral_recall": recalls[1],
        "positive_recall": recalls[2],
    })

    print(f"\n{name}: rows=true, columns=predicted")
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=[0, 1, 2]),
        index=label_names,
        columns=label_names,
    ))

summary_df = pd.DataFrame(summary).set_index("model")

print("\nFull validation comparison across three models:")
display(summary_df.round(4))

summary_df.to_csv(
    save_dir / "validation_comparison_with_jev.csv"
)


Base model: rows=true, columns=predicted


,negative,neutral,positive
negative,234,10,68
neutral,255,144,470
positive,60,35,724



LoRA model: rows=true, columns=predicted


,negative,neutral,positive
negative,289,9,14
neutral,300,347,222
positive,68,58,693



Jev 1.13.0: rows=true, columns=predicted


,negative,neutral,positive
negative,278,23,11
neutral,246,391,232
positive,64,92,663



Full validation comparison across three models:


,accuracy,macro-F1,negative_recall,neutral_recall,positive_recall
model,,,,,
Base model,0.5510,0.5039,0.7500,0.1657,0.8840
LoRA model,0.6645,0.6434,0.9263,0.3993,0.8462
Jev 1.13.0,0.6660,0.6517,0.8910,0.4499,0.8095


In [23]:
# Keep LoRA enabled for training and test split evaluation.
assert len(train_small) == 9000

save_dir.mkdir(parents=True, exist_ok=True)

results_train_lora = evaluate_validation(
    "LoRA / train",
    backend="qwen",
    dataset=train_small,
)
results_train_lora.to_csv(
    save_dir / "train_lora.csv", index=False
)

results_test_lora = evaluate_validation(
    "LoRA / test",
    backend="qwen",
    dataset=ds["test"],
)
results_test_lora.to_csv(
    save_dir / "test_lora.csv", index=False
)

LoRA / train:   0%|          | 0/9000 [00:00<?, ?it/s]

LoRA / test:   0%|          | 0/12284 [00:00<?, ?it/s]

In [24]:
summary = []

for split, frame in [
    ("train", results_train_lora),
    ("validation", results_lora),
    ("test", results_test_lora),
]:
    y_true = frame["true_label"]
    y_pred = frame["pred_label"]

    recalls = recall_score(
        y_true, y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0,
    )

    summary.append({
        "split": split,
        "examples": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro-F1": f1_score(
            y_true, y_pred,
            labels=[0, 1, 2],
            average="macro",
            zero_division=0,
        ),
        "negative_recall": recalls[0],
        "neutral_recall": recalls[1],
        "positive_recall": recalls[2],
    })

split_summary = pd.DataFrame(summary).set_index("split")
display(split_summary.round(4))

split_summary.to_csv(save_dir / "lora_split_comparison.csv")

,examples,accuracy,macro-F1,negative_recall,neutral_recall,positive_recall
split,,,,,,
train,9000,0.7584,0.7399,0.9593,0.4477,0.8683
validation,2000,0.6645,0.6434,0.9263,0.3993,0.8462
test,12284,0.6241,0.6238,0.9615,0.3185,0.8236
